In [1]:
!apt-get install openjdk-17-jdk-headless -qq > /dev/null # Instalar SDK java 17

In [2]:
!wget -q https://downloads.apache.org/spark/spark-4.0.2/spark-4.0.2-bin-hadoop3.tgz # Descargar Spark 4.0.1

In [3]:
!tar xf spark-4.0.2-bin-hadoop3.tgz # Descomprimir la version de Spark

In [4]:
!pip install -q findspark # Instalar la librería findspark

In [5]:
!pip install -q pyspark # Instalar pyspark

# Funciones

## Funciones creadas por el usuario UDF

In [6]:
import findspark
findspark.init()
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

In [7]:
def f_cubo(n):
  return n * n * n

In [8]:
from pyspark.sql.types import LongType

In [9]:
spark.udf.register('cubo', f_cubo, LongType());

In [10]:
spark.range(1, 10).createOrReplaceTempView('df_temp')

In [11]:
spark.sql("SELECT id, cubo(id) AS cubo FROM df_temp").show() # Elemento ID al cubo

+---+----+
| id|cubo|
+---+----+
|  1|   1|
|  2|   8|
|  3|  27|
|  4|  64|
|  5| 125|
|  6| 216|
|  7| 343|
|  8| 512|
|  9| 729|
+---+----+



In [12]:
def bienvenida(nombre):
  return ('Hola {}'.format(nombre))

In [13]:
from pyspark.sql.functions import udf
from pyspark.sql.functions import StringType

In [14]:
bienvenida_udf = udf(lambda x: bienvenida(x), StringType())

In [15]:
# bienvenida_udf

In [16]:
from pyspark.sql.types import StructType, StructField, StringType
from pyspark.sql import Row

schema = StructType([StructField("nombre", StringType(), True)])
data = [Row("Jose"), Row("Julia")] # Use Row objects for clearer structure
df_nombre = spark.createDataFrame(data, schema=schema)

In [17]:
df_nombre.show()

+------+
|nombre|
+------+
|  Jose|
| Julia|
+------+



In [18]:
from pyspark.sql.functions import col

In [19]:
df_nombre.select(
    col('nombre'),
    bienvenida_udf(col('nombre')).alias('saludo')
).show()

+------+----------+
|nombre|    saludo|
+------+----------+
|  Jose| Hola Jose|
| Julia|Hola Julia|
+------+----------+



In [20]:
@udf(returnType=StringType())
def mayuscula(s):
  return s.upper()

In [21]:
df_nombre.select(
    col('nombre'),
    mayuscula(col('nombre')).alias('mayuscula')
).show()

+------+---------+
|nombre|mayuscula|
+------+---------+
|  Jose|     JOSE|
| Julia|    JULIA|
+------+---------+



In [22]:
import pandas as pd
from pyspark.sql.functions import pandas_udf

In [23]:
def cubo_pandas(a: pd.Series) -> pd.Series:
  return a * a * a

In [24]:
cubo_udf = pandas_udf(cubo_pandas, returnType=LongType())

In [25]:
x = pd.Series([1, 2, 3])

In [26]:
print(cubo_pandas(x))

0     1
1     8
2    27
dtype: int64


## Funciones de ventana

In [30]:
import findspark
findspark.init()
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()
df =  spark.read.parquet('/content/drive/MyDrive/RDD_FILES/sparksql/datos.csv/funciones_ventana.parquet')

In [31]:
df.show()

+-------+----+------------+----------+
| nombre|edad|departamento|evaluacion|
+-------+----+------------+----------+
| Lazaro|  45|      letras|        98|
|   Raul|  24|  matemática|        76|
|  Maria|  34|  matemática|        27|
|   Jose|  30|     química|        78|
| Susana|  51|     química|        98|
|   Juan|  44|      letras|        89|
|  Julia|  55|      letras|        92|
|  Kadir|  38|arquitectura|        39|
| Lilian|  23|arquitectura|        94|
|   Rosa|  26|      letras|        91|
|   Aian|  50|  matemática|        73|
|Yaneisy|  29|      letras|        89|
|Enrique|  40|     química|        92|
|    Jon|  25|arquitectura|        78|
|  Luisa|  39|arquitectura|        94|
+-------+----+------------+----------+



In [32]:
from pyspark.sql.window import Window
from pyspark.sql.functions import desc, row_number, rank, dense_rank

In [33]:
WindowSpec = Window.partitionBy('departamento').orderBy(desc('evaluacion'))

In [34]:
df.withColumn('row_number', row_number().over(WindowSpec)).show()

+-------+----+------------+----------+----------+
| nombre|edad|departamento|evaluacion|row_number|
+-------+----+------------+----------+----------+
| Lilian|  23|arquitectura|        94|         1|
|  Luisa|  39|arquitectura|        94|         2|
|    Jon|  25|arquitectura|        78|         3|
|  Kadir|  38|arquitectura|        39|         4|
| Lazaro|  45|      letras|        98|         1|
|  Julia|  55|      letras|        92|         2|
|   Rosa|  26|      letras|        91|         3|
|   Juan|  44|      letras|        89|         4|
|Yaneisy|  29|      letras|        89|         5|
|   Raul|  24|  matemática|        76|         1|
|   Aian|  50|  matemática|        73|         2|
|  Maria|  34|  matemática|        27|         3|
| Susana|  51|     química|        98|         1|
|Enrique|  40|     química|        92|         2|
|   Jose|  30|     química|        78|         3|
+-------+----+------------+----------+----------+



In [35]:
df.withColumn('row_number', row_number().over(WindowSpec)).filter(col('row_number').isin(1, 2)).show()

+-------+----+------------+----------+----------+
| nombre|edad|departamento|evaluacion|row_number|
+-------+----+------------+----------+----------+
| Lilian|  23|arquitectura|        94|         1|
|  Luisa|  39|arquitectura|        94|         2|
| Lazaro|  45|      letras|        98|         1|
|  Julia|  55|      letras|        92|         2|
|   Raul|  24|  matemática|        76|         1|
|   Aian|  50|  matemática|        73|         2|
| Susana|  51|     química|        98|         1|
|Enrique|  40|     química|        92|         2|
+-------+----+------------+----------+----------+



In [36]:
df.withColumn('rank', rank().over(WindowSpec)).show()

+-------+----+------------+----------+----+
| nombre|edad|departamento|evaluacion|rank|
+-------+----+------------+----------+----+
| Lilian|  23|arquitectura|        94|   1|
|  Luisa|  39|arquitectura|        94|   1|
|    Jon|  25|arquitectura|        78|   3|
|  Kadir|  38|arquitectura|        39|   4|
| Lazaro|  45|      letras|        98|   1|
|  Julia|  55|      letras|        92|   2|
|   Rosa|  26|      letras|        91|   3|
|   Juan|  44|      letras|        89|   4|
|Yaneisy|  29|      letras|        89|   4|
|   Raul|  24|  matemática|        76|   1|
|   Aian|  50|  matemática|        73|   2|
|  Maria|  34|  matemática|        27|   3|
| Susana|  51|     química|        98|   1|
|Enrique|  40|     química|        92|   2|
|   Jose|  30|     química|        78|   3|
+-------+----+------------+----------+----+



In [37]:
df.withColumn('dense_rank', dense_rank().over(WindowSpec)).show()


+-------+----+------------+----------+----------+
| nombre|edad|departamento|evaluacion|dense_rank|
+-------+----+------------+----------+----------+
| Lilian|  23|arquitectura|        94|         1|
|  Luisa|  39|arquitectura|        94|         1|
|    Jon|  25|arquitectura|        78|         2|
|  Kadir|  38|arquitectura|        39|         3|
| Lazaro|  45|      letras|        98|         1|
|  Julia|  55|      letras|        92|         2|
|   Rosa|  26|      letras|        91|         3|
|   Juan|  44|      letras|        89|         4|
|Yaneisy|  29|      letras|        89|         4|
|   Raul|  24|  matemática|        76|         1|
|   Aian|  50|  matemática|        73|         2|
|  Maria|  34|  matemática|        27|         3|
| Susana|  51|     química|        98|         1|
|Enrique|  40|     química|        92|         2|
|   Jose|  30|     química|        78|         3|
+-------+----+------------+----------+----------+



In [38]:
windowSpecAgg = Window.partitionBy('departamento')

In [45]:
windowSpec = Window.partitionBy('departamento').orderBy(desc('evaluacion'))

## Función compuesta por muchas funciones de agreación

In [39]:
from pyspark.sql.functions import col, avg, sum, min, max, count

In [47]:
(df.withColumn('min', min(col('evaluacion')).over(windowSpecAgg))
.withColumn('max', max(col('evaluacion')).over(windowSpecAgg))
.withColumn('avg', avg(col('evaluacion')).over(windowSpecAgg))
.withColumn('row_number', row_number().over(windowSpec))
 ).show()

+-------+----+------------+----------+---+---+------------------+----------+
| nombre|edad|departamento|evaluacion|min|max|               avg|row_number|
+-------+----+------------+----------+---+---+------------------+----------+
| Lilian|  23|arquitectura|        94| 39| 94|             76.25|         1|
|  Luisa|  39|arquitectura|        94| 39| 94|             76.25|         2|
|    Jon|  25|arquitectura|        78| 39| 94|             76.25|         3|
|  Kadir|  38|arquitectura|        39| 39| 94|             76.25|         4|
| Lazaro|  45|      letras|        98| 89| 98|              91.8|         1|
|  Julia|  55|      letras|        92| 89| 98|              91.8|         2|
|   Rosa|  26|      letras|        91| 89| 98|              91.8|         3|
|   Juan|  44|      letras|        89| 89| 98|              91.8|         4|
|Yaneisy|  29|      letras|        89| 89| 98|              91.8|         5|
|   Raul|  24|  matemática|        76| 27| 76|58.666666666666664|         1|